#  The Haunted Pendulum
## A Physics Lab on Chaos and Strange Attractors

Welcome to the lab. Today we're investigating a system that *looks* simple but hides a deep and complex secret. We're modeling a **damped, driven pendulum**.

This is a single pendulum that is being pushed by a motor (the **drive**) and slowed down by friction (the **damping**). By tuning these forces, we can make it swing in a simple, predictable pattern... or make it behave in a way that is completely erratic and unpredictable.

This unpredictable behavior is known as **deterministic chaos**.

### What is Chaos?
Chaos is **not** complete randomness. It is:
1.  **Deterministic:** The motion is governed by exact, unchanging physical laws (Newton's laws, in this case). There is no random number generator.
2.  **Aperiodic:** The motion never repeats itself, ever.
3.  **Sensitive:** It shows "sensitive dependence on initial conditions." This is the "Butterfly Effect". A tiny, microscopic change in the starting position will lead to a completely different, unpredictable future.

### Why Study This "Haunted" Pendulum?
Chaos is everywhere in nature: the weather, the orbits of planets in our solar system, the turbulence of a flowing stream, and the fluctuations of animal populations.

These systems are far too complex to study from scratch. Instead, physicists study simple "toy models" like this pendulum. They discovered that the *way* this pendulum becomes chaotic (a process called **period-doubling**) is a **universal** law. The same pattern it follows is seen in electrical circuits, fluid dynamics, and biological systems.

By understanding this one "haunted" pendulum, we unlock a fundamental secret of how chaos works everywhere.

Some additional notes about plots you'll be producing.

### Phase Space & Phase Plots

In mechanics, the "state" of a simple system (like a 1D particle) isn't just its position $x$. To know its future, you also need to know its momentum $p$ (or velocity $v$).

* **Configuration Space:** The space of all possible *positions* (e.g., just the $x$-axis).
* **Phase Space:** The space of all possible *states*. For a 1D particle, this is a 2D space with coordinates $(x, p)$. A single point in phase space defines the complete state of the system at a specific instant.

A **phase plot** (or phase portrait) visualizes the system's evolution. As the system evolves in time, its state $(x(t), p(t))$ traces a path in phase space. This path is called a **trajectory**.

A phase plot shows a collection of these trajectories starting from different initial conditions.

**Example: The Simple Harmonic Oscillator (SHO)**

The equation of motion is $\ddot{x} = -\omega^2 x$.
The system's state is $(x, v)$, where $v = \dot{x}$.
The total energy $E = \frac{1}{2}mv^2 + \frac{1}{2}kx^2$ is conserved.
Rearranging the energy equation gives:
$$\frac{x^2}{(2E/k)} + \frac{v^2}{(2E/m)} = 1$$
This is the equation of an **ellipse**.

* **What the plot shows:** In $(x, v)$ phase space, the trajectory of an SHO is a closed ellipse.
* **Interpretation:**
    * The system is **periodic**: It always returns to its starting state.
    * Different initial conditions (different energies) correspond to different ellipses.
    * The plot captures the *entire* set of possible behaviors (a set of nested ellipses) at a glance.



For a **damped** oscillator, the energy is *not* conserved. The trajectory is a spiral that winds inward toward the origin $(0, 0)$, which is a stable fixed point (or "attractor").

---

### Poincaré Plots (Poincaré Sections)

Phase plots are great, but they get very complicated.
* An SHO (1 degree of freedom) has a 2D phase space.
* A 3D system like the Lorenz attractor (3 variables) has a 3D phase space.
* A double pendulum (2 degrees of freedom) has a 4D phase space $(q_1, q_2, p_1, p_2)$.

We can't visualize 3D flows (they look like a tangled mess of yarn) or 4D spaces at all.

A **Poincaré plot** is a brilliant trick to simplify the visualization. It reduces the dimension of the system, acting like a "stroboscope" that samples the system's trajectory.

**How it works:**
1.  You start with your continuous trajectory in an $N$-dimensional phase space.
2.  You choose a specific $(N-1)$-dimensional "hyperplane" (a "Poincaré section") that cuts through the space.
3.  You let the system evolve, but you *only* plot a dot **every time the trajectory passes through the section in a specific direction** (e.g., only when $x$ is increasing).

**Analogy:** Imagine a tube of toothpaste (the trajectory) in 3D space. A Poincaré plot is the set of 2D dots you'd get by repeatedly slicing the tube with a sheet of paper (the Poincaré section).



**Why is this so useful?**
It "freezes" the dynamics and reveals the underlying structure. It turns a continuous flow into a discrete map.

**Interpreting a Poincaré Plot:**

* **Periodic Orbit:** A simple, repeating loop in phase space (like the SHO).
    * **Poincaré Plot:** A **single point** (or a small, finite set of $N$ points if the orbit is more complex but still periodic). The trajectory hits the same spot on the section every single time.

* **Quasi-periodic Orbit:** A more complex, non-repeating orbit that *never* closes on itself. (Think of two independent frequencies, like $\sin(t) + \sin(\pi t)$). The trajectory densely fills the surface of a 2D torus in phase space.
    * **Poincaré Plot:** A **closed loop** or **ring**. Slicing a torus gives you a circle.

* **Chaotic Orbit:** A non-repeating, unpredictable orbit that is highly sensitive to initial conditions. It's confined to a region called a "strange attractor."
    * **Poincaré Plot:** A complex, fractal-like pattern of dots. It's not a single point, not a simple curve, but an intricate structure that hints at the "stretching and folding" nature of chaos. The Lorenz attractor's Poincaré plot is a famous example.

In short, Poincaré plots are an essential tool for "seeing" the difference between simple periodic motion, complex but predictable quasi-periodic motion, and true chaos.

### Importing the Libraries
This cell imports the Python libraries we need.
* `numpy`: For all the numerical calculations.
* `matplotlib`: For plotting our results.
* `scipy`: We import its solver (`solve_ivp`) to see how our custom-built solver compares.
* `ipywidgets`: To create the interactive control panel.
* `numba`: This is our new **computational physics tool**. It's a Just-In-Time (JIT) compiler that will make our Python solver run *hundreds* of times faster.
* `tqdm`: A helpful library for creating progress bars for our loops. **bold text**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout
from numba import jit
from tqdm.auto import tqdm

### The Laws of Motion & Our Solver

This is our main computational physics challenge for today. We are going to **build our own ODE solver from scratch**. This cell is split into two parts:

1.  **Part A: The Physics (`derivs`):** The function that defines the equations of motion.
2.  **Part B: The Solver (`rk4_step`):** The function that *solves* those equations step-by-step.

---

#### Part A: The Physics (The `derivs` function)

First, we define the physics "engine." The equation of motion is:
$$\frac{d^2\theta}{dt^2} = -q \frac{d\theta}{dt} - \sin(\theta) + F_D \cos(\omega_D t)$$

We add a `@jit(nopython=True)` decorator. This is a **Just-In-Time (JIT) compiler** from the `numba` library. It will analyze our Python function and compile it down to highly optimized machine code, making it run *hundreds* of times faster than standard Python. This is a critical technique in computational physics.

**Your Task:**
Fill in the line for `d_omega_dt` in the `derivs` code cell below, based on the equation of motion above. (Remember that $\frac{d^2\theta}{dt^2}$ is `d_omega_dt` and $\frac{d\theta}{dt}$ is `omega`).

---
#### A Note for the Curious: "Where is gravity?"

You might be looking at that equation and thinking, "Shouldn't it be $-\frac{g}{L}\sin(\theta)$? And what are `q` and $F_D$ and $\omega$?

The equation you see above is a **"non-dimensionalized"** version. It's a trick to make the math cleaner.

Here’s what's happening:

1.  **Where is gravity? ($-\frac{g}{L} \sin(\theta)$ vs. $-\sin(\theta)$)**
    Gravity isn't gone. We've changed our unit of time. Instead of measuring time in **seconds**, we measure it in **"natural units"** based on the pendulum's own rhythm ($\sqrt{L/g}$). When we rewrite the equation using this new, "natural" time, the entire $\frac{g}{L}$ term mathematically simplifies to be **exactly 1**. So, the `1` implied in front of $-\sin(\theta)$ *is* the gravity term in this new system.

2.  What is `q`?
     It is a damping coeffient that takes into account friction, air resistance, and other effects.
    
2.  What is `$F_D$`?
     It is a driving force.

2.  What is `$\omega$`?
     It is a driving frequency.

Setting the equation up this way lets us explore the *fundamental behavior* of the system. We can make a universal statement like "Chaos appears when `F_D` is 1.2" without having to say "...for a pendulum of length 0.8m, on a planet with gravitational acceleration of ...

---

### A Note on  `@jit`
You'll add the `@jit(nopython=True)` decorator above your deriv and solver functions. This isn't just a comment; it's an instruction that unlocks a huge performance boost. This is one of the most important "tricks" in computational physics: **Just-In-Time (JIT) compilation**.

#### What is Numba?
`numba` is a JIT compiler for Python.

#### What is a JIT Compiler (and why do we need it)?

1.  **The Problem:** Python is an "interpreted" language. When you run a `for` loop, the interpreter has to check the type of every variable *every single time* through the loop ("is `y` still a number? is `dt` still a number?"). This adds a massive amount of overhead, making Python *very slow* for heavy-duty numerical loops (like our RK4 solver, which runs millions of times).

2.  **The Solution:** A "Just-In-Time" (JIT) compiler, like `numba`, acts as a translator.
    * **The First Time:** The *first time* you call your `rk4_step` function, `numba` watches it run. It "infers" the types (e.g., "Ah, `t` is a float, `y` is a NumPy array of floats...").
    * **It Compiles:** It then translates your Python code into highly optimized machine code (like C or Fortran) specifically for those types. This translation takes a second or two.
    * **Every Other Time:** From then on, every time you call `rk4_step`, Python just runs that super-fast, pre-compiled machine code.

**The Result:** Your first simulation run will have a slight delay (for compilation). Every run after that will be **100-1000x faster**, allowing us to use the interactive sliders in real-time. We get the high performance of C without having to leave Python.

###The Laws of Motion & Our Solver

This is our main computational physics challenge for today. We are going to **build our own ODE solver from scratch**. This cell is split into two parts:

1.  **Part A: The Physics (`derivs`):** The function that defines the equations of motion.
2.  **Part B: The Solver (`rk4_step`):** The function that *solves* those equations step-by-step (which we'll do in the *next* markdown cell).

---

#### The Physics (The `derivs` function)

Before we can build a solver, we must define the physics. Our equation of motion is a 2nd-order Ordinary Differential Equation (ODE):
$$\frac{d^2\theta}{dt^2} = -q \frac{d\theta}{dt} - \sin(\theta) + F_D \cos(\omega_D t)$$

Our solver (RK4) can't solve one 2nd-order equation. It can only solve a *system* of 1st-order equations. We must first convert it.

We do this by defining a **state vector** `state = [theta, omega]`, where $\omega$ is the angular velocity ($\omega = \frac{d\theta}{dt}$).

Our `derivs` function must take this state and return its derivative: `[d_theta_dt, d_omega_dt]`.

This gives us our two 1st-order equations:
1.  `d_theta_dt` = $\omega$
2.  `d_omega_dt` = $-q \omega - \sin(\theta) + F_D \cos(\omega_D t)$

**Your Task:**

In the code cell below, complete the `derivs` function by:
1.  **Unpacking** the `state` array into its two components, `theta` and `omega`.
2.  **Setting** the variable `d_theta_dt` using **Equation 1**.
3.  **Setting** the variable `d_omega_dt` using **Equation 2**.

(We've already added the `@jit` decorator so `numba` can compile this for us, which is why we'll return a `np.array`.)

In [ ]:
@jit(nopython=True) # <-- This decorator tells Numba to compile this function
def derivs(t, state, q, F_D, w_D):
    """
    Computes the derivatives for the damped, driven pendulum.
    The state is a NumPy array: [theta, omega]
    """
    theta, omega = state[0], state[1]

    d_theta_dt = omega

    # TODO : Fill in the differential equation for d_omega_dt
    # Look at the lab description in Cell 2 for the equation
    d_omega_dt = (-q*omega) - np.sin(theta) + (F_D * np.cos(w_D*t))

    # Numba works fastest with NumPy arrays
    return np.array([d_theta_dt, d_omega_dt])

#### The Solver (The `rk4_step` function)

Now, we'll build the *solver* itself. Instead of the simple (and often unstable) Euler method, we'll use the gold standard for many physics problems: the **4th-Order Runge-Kutta (RK4) algorithm**.

This method is much more accurate and stable because it cleverly samples the derivative at four different points within the time step ($dt$) and then takes a weighted average.

**The Algorithm (Our Recipe):**

The algorithm is a step-by-step recipe. To get from our current state $\vec{y}_n$ to the next state $\vec{y}_{n+1}$, we first calculate four intermediate "slopes" ($k$-values):

1.  $\vec{k}_1 = dt \cdot f(t_n, \vec{y}_n)$
2.  $\vec{k}_2 = dt \cdot f(t_n + \frac{dt}{2}, \vec{y}_n + \frac{\vec{k}_1}{2})$
3.  $\vec{k}_3 = dt \cdot f(t_n + \frac{dt}{2}, \vec{y}_n + \frac{\vec{k}_2}{2})$
4.  $\vec{k}_4 = dt \cdot f(t_n + dt, \vec{y}_n + \vec{k}_3)$

And the final step is a weighted average of these slopes:

$\vec{y}_{n+1} = \vec{y}_n + \frac{1}{6}(\vec{k}_1 + 2\vec{k}_2 + 2\vec{k}_3 + \vec{k}_4)$

---

**Your Task:** Translate this "recipe" into code in the cell below.

* `f(...)` is our `derivs(...)` function.
* `y` (or $\vec{y}$) is the state array `[theta, omega]`.
* `t` is the current time $t_n$.

**Instructions:**

1.  **TODO (The 'k' steps):** Fill in the lines for `k1`, `k2`, `k3`, and `k4` by translating the math equations above into code.
    * Remember that `y`, `k1`, `k2`, etc., are NumPy arrays, so `y + k1/2` is a simple array addition.

2.  **TODO (The final step):** Fill in the line for `y_next` using the final weighted average equation.

3.  **TODO (The Critical Angle Wrap):** This is a crucial step for this *specific* problem. Angles are periodic ($361^\circ$ is the same as $1^\circ$). If we let $\theta$ (the first element of `y_next`, or `y_next[0]`) grow to thousands of radians, our $\sin(\theta)$ calculation will become numerically unstable and the simulation will fail.

    * To fix this, we must "wrap" the angle after every step to keep it between $-\pi$ and $+\pi$.
    * Uncomment and complete this line of code using the provided formula:
        `y_next[0] = (y_next[0] + np.pi) % (2 * np.pi) - np.pi`

In [ ]:
@jit(nopython=True)
def rk4_step(t, y, dt, q, F_D, w_D):
    """Performs a single 4th-Order Runge-Kutta step."""

    # TODO : Implement the four RK4 "k" steps using the equations above.
    # y is the state array [theta, omega]
    # Our "f" function is derivs()

    k1 = dt * derivs(t, y, q, F_D, w_D)
    k2 = dt * derivs(t + (0.5 * dt), y + (0.5 * k1), q, F_D, w_D)
    k3 = dt * derivs(t + (0.5 * dt), y + (0.5 * k2), q, F_D, w_D)
    k4 = dt * derivs(t + dt, y + k3, q, F_D, w_D)

    # TODO : Combine the k-values to get the next state, y_next
    y_next = y + ((k1 + 2*k2 + 2*k3 + k4)/6)

    # TODO : CRITICAL STEP! Wrap the angle (theta)
    # This keeps the simulation stable.
    # The angle is the *first* element of the y_next array (index 0).
    y_next[0] = (y_next[0] + np.pi) % (2 * np.pi) - np.pi


    return y_next

### The Control Panel 🎛️

This cell creates the interactive control panel for our experiment.

It defines two main types of objects:
1.  **Sliders & Button:** These are our "knobs" and "switches." We've added a new one, `Pts per Cycle`, which is a crucial part of our new custom solver.
2.  **Output Areas:** These are the four blank "canvases" where our plots (`Time Series`, `FFT`, `Poincaré`, and `Phase Space`) will appear.

**The Controls:**
* **q**: The damping coefficient (friction).
* **F_D**: The drive strength. **This is the key to chaos!**
* **w_D**: The drive frequency (how fast the motor pushes).
* **Start θ (deg)**: The initial starting angle of the pendulum.
* **N Cycles**: How many drive cycles to simulate.
* **Pts per Cycle (NEW):** This is our **solver accuracy knob!** It sets the number of calculation steps ($dt$) our new RK4 solver will perform *per* drive cycle. A higher number means more accuracy.
* **Run Simulation Button:** The button that starts the physics engine.

In [ ]:
# Define layout for sliders
slider_layout = Layout(width='300px')

# Sliders for physical properties
q_slider = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description='Damping (q):', layout=slider_layout)
FD_slider = widgets.FloatSlider(value=1.2, min=0.0, max=1.5, step=0.01, description='Drive (F_D):', layout=slider_layout)
wD_slider = widgets.FloatSlider(value=0.667, min=0.1, max=2.0, step=0.001, description='Drive Freq (ω_D):', layout=slider_layout)

# Sliders for initial conditions
th0_slider = widgets.FloatSlider(value=0.0, min=-180, max=180, step=1, description='Start θ (deg):', layout=slider_layout)

# Simulation parameters
n_cycles_slider = widgets.IntSlider(value=300, min=50, max=1000, step=10, description='N Cycles:', layout=slider_layout)
# --- NEW SLIDER ---
pts_per_cycle_slider = widgets.IntSlider(value=100, min=20, max=500, step=10, description='Pts per Cycle:', layout=slider_layout)


# Button to run
run_button = widgets.Button(description="Run Simulation", button_style='success')

# Output areas
out_timeseries = widgets.Output(layout={'border': '1px solid black', 'width': '97%'})
out_fft = widgets.Output(layout={'border': '1px solid black', 'width': '97%'})
out_poincare = widgets.Output(layout={'border': '1px solid black', 'width': '48%'})
out_phase = widgets.Output(layout={'border': '1px solid black', 'width': '48%'})

### The Simulation Loop

This is the main function, `run_simulation`, that runs when you click the "Run Simulation" button. This is where all our work from Cell 2 comes together.

Instead of calling `solve_ivp`, this function will:
1.  Grab all the values from your sliders (like $F_D$, $q$, and our new `Pts per Cycle`).
2.  Calculate the time step, `dt`, from your `Pts per Cycle` slider.
3.  Create empty NumPy arrays (`t_vals`, `theta_vals`, etc.) to store the results.
4.  Run a large `for` loop (wrapped in `tqdm` to give us a progress bar) that calls your new **`rk4_step`** function for every time step.
5.  As it runs, it will store *all* the results for the Time Series and Phase Space plots.
6.  It will *also* use an `if` statement to intelligently save points *only* for the **Poincaré Plot** (the "strobe light" snapshot).
7.  Finally, it will use all this collected data to generate our four plots.

---
**Your Task:**

This function is almost complete. You just need to fill in the two `TODO`s inside the main solver loop.

1.  **TODO  (The Solver Step):** Find the `for i in tqdm(range(n_points))` loop. Inside it, you need to:
    * Call your `rk4_step` function (from Cell 2) to calculate the `y_next` state.
    * Update the current state `y` to be this new `y_next`.
    * Advance the current time `t` by one `dt`.

2.  **TODO (The Poincaré Logic):** Find the `if` statement. You need to write the condition that checks for two things *at the same time*:
    * That we are past the transient phase (i.e., `i >= skip_steps`).
    * That we are at the *start* of a drive cycle (i.e., `i % points_per_cycle == 0`).

In [ ]:
# This is the "main" function that runs when you click the button.
def run_simulation(b):
    # Clear all outputs
    out_timeseries.clear_output(wait=True)
    out_fft.clear_output(wait=True)
    out_poincare.clear_output(wait=True)
    out_phase.clear_output(wait=True)

    # --- Get values from sliders ---
    q = q_slider.value
    F_D = FD_slider.value
    w_D = wD_slider.value
    th0_rad = np.deg2rad(th0_slider.value)
    w0 = 0.0
    N_cycles = n_cycles_slider.value
    points_per_cycle = pts_per_cycle_slider.value

    # --- Time Setup ---
    T_D = 2.0 * np.pi / w_D
    dt = T_D / points_per_cycle # Time step is now user-controlled
    n_points = N_cycles * points_per_cycle + 1
    y0_np = np.array([th0_rad, w0])
    skip_steps = 50 * points_per_cycle # Skip 50 cycles for transient

    # --- Create result arrays ---
    t_vals = np.zeros(n_points)
    theta_vals = np.zeros(n_points)
    omega_vals = np.zeros(n_points)

    # Pre-allocate arrays for Poincaré points
    # We will save at most (N_cycles - 50) points
    poincare_theta = np.zeros(max(0, N_cycles - 50))
    poincare_omega = np.zeros(max(0, N_cycles - 50))

    # --- Run the Solver Loop ---
    y = y0_np
    t = 0.0
    poincare_count = 0

    print("Running RK4 Solver...")

    # We use tqdm() to make a nice progress bar
    for i in tqdm(range(n_points)):

        # --- TODO: The main solver step ---
        # Call the rk4_step function to get the next state `y`
        # and advance the time `t` by one `dt`

        y = rk4_step(t, y, dt, q, F_D, w_D)
        t = t + dt

        # --- Store full results ---
        t_vals[i] = t
        theta_vals[i] = y[0]
        omega_vals[i] = y[1]

        # --- TODO: Poincaré Plot Logic ---
        # We need to save a point *only* if:
        # 1. We are past the transient phase (i >= skip_steps)
        # 2. We are at the *start* of a drive cycle (i % points_per_cycle == 0)

        if (i >= skip_steps) and (i % points_per_cycle == 0):
            if poincare_count < len(poincare_theta): # Avoid array overflow
                poincare_theta[poincare_count] = y[0]
                poincare_omega[poincare_count] = y[1]
                poincare_count += 1

    print("...Done!")

    # --- Prune arrays to the actual number of points saved ---
    start_index = skip_steps
    poincare_theta = poincare_theta[:poincare_count]
    poincare_omega = poincare_omega[:poincare_count]

    # --- Generate Time Series Plot ---
    with out_timeseries:
        t_steady = t_vals[start_index:]
        theta_steady_wrapped = theta_vals[start_index:] # Already wrapped

        fig_t, ax_t = plt.subplots(figsize=(9.5, 3))
        ax_t.plot(t_steady, theta_steady_wrapped, 'b-', lw=0.5)
        ax_t.set_title(r'Time Series $\theta(t)$ (after transient)')
        ax_t.set_xlabel('Time (dimensionless)')
        ax_t.set_ylabel(r'$\theta$ (rad)')
        ax_t.set_xlim(t_steady[0], t_steady[-1])
        ax_t.set_ylim(-np.pi, np.pi)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.show(fig_t)

    # --- Generate Fourier Transform (FFT) Plot ---
    with out_fft:
        n_points_steady = len(t_steady)
        theta_steady = theta_vals[start_index:]

        fft_vals = np.fft.fft(theta_steady - np.mean(theta_steady))
        fft_freq = np.fft.fftfreq(n_points_steady, d=dt)
        fft_power = np.abs(fft_vals)**2

        mask = fft_freq > 0
        fig_f, ax_f = plt.subplots(figsize=(9.5, 3))
        ax_f.plot(fft_freq[mask], fft_power[mask], 'r-', lw=0.5)
        ax_f.set_title('Frequency Spectrum (FFT)')
        ax_f.set_xlabel('Frequency (f, dimensionless)')
        ax_f.set_ylabel('Power')

        f_D = w_D / (2 * np.pi)
        ax_f.set_xlim(0, f_D * 4)
        ax_f.set_ylim(bottom=1e-1)

        ax_f.axvline(f_D, color='k', linestyle='--', lw=1, label=f'$f_D$ = {f_D:.2f}')
        ax_f.axvline(f_D / 2, color='b', linestyle=':', lw=1, label='$f_D/2$')
        ax_f.axvline(f_D / 4, color='g', linestyle=':', lw=1, label='$f_D/4$')

        ax_f.set_yscale('log')
        ax_f.legend(fontsize='small')
        ax_f.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show(fig_f)

    # --- Generate Poincaré Plot ---
    with out_poincare:
        fig_p, ax_p = plt.subplots(figsize=(6, 6))
        ax_p.plot(poincare_theta, poincare_omega, 'k.', ms=1.5, alpha=0.8)
        ax_p.set_title(fr'Poincaré Section (Stroboscopic Plot)')
        ax_p.set_xlabel(r'$\Theta$ (rad) at $t = n \cdot T_D$')
        ax_p.set_ylabel(r'$\omega$ (rad/s) at $t = n \cdot T_D$')
        ax_p.set_xlim(-np.pi, np.pi)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.show(fig_p)

    # --- Generate Phase Space Plot ---
    with out_phase:
        theta_full = theta_vals[start_index:] # Already wrapped
        omega_full = omega_vals[start_index:]

        fig_ph, ax_ph = plt.subplots(figsize=(6, 6))
        ax_ph.plot(theta_full, omega_full, 'b-', lw=0.1, alpha=0.5)
        ax_ph.set_title(r'Phase Space (Full Trajectory)')
        ax_ph.set_xlabel(r'$\Theta$ (rad)')
        ax_ph.set_ylabel(r'$\omega$ (rad/s)')
        ax_ph.set_xlim(-np.pi, np.pi)
        ax_ph.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show(fig_ph)


# Attach the function to the button
run_button.on_click(run_simulation)

### The simulation interface

This final cell displays the interface. **Run this cell**, and your control panel will appear.

Once it appears, follow the lab instructions below.

In [ ]:
# Arrange the controls
controls_row1 = HBox([q_slider, FD_slider])
controls_row2 = HBox([wD_slider, th0_slider])
controls_row3 = HBox([n_cycles_slider, pts_per_cycle_slider, run_button]) # Added new slider
controls = VBox([controls_row1, controls_row2, controls_row3])

# Arrange the outputs
plot_outputs = HBox([out_poincare, out_phase])

# Display the final UI
display(VBox([controls, out_timeseries, out_fft, plot_outputs]))

Running RK4 Solver...


  0%|          | 0/60001 [00:00<?, ?it/s]

...Done!


# Your Lab Experiments

Your goal is to use the plots to find the "ghost in the machine"—the hidden, chaotic pattern. We will do this by slowly turning up the **Drive Strength (F_D)**, which is our "chaos knob."

*(Use the default values for `q` and `ω_D` unless told otherwise. Run `N Cycles` for 300+ to see the patterns clearly.)*

---

### Experiment 0: Test Your Engine! (Solver Accuracy)

In Cell 2, you built an RK4 solver. Its accuracy depends on the time step `dt`, which you control with **`Pts per Cycle`**. Let's see what happens if we're not careful.

1.  Set up the **Chaotic State**:
    * **Drive (F_D): 1.2**
    * **Pts per Cycle: 20** (the minimum)
2.  Click **"Run Simulation"** and look at the Poincaré Plot.
3.  Now, set:
    * **Pts per Cycle: 200** (a high-accuracy value)
4.  Click **"Run Simulation"** again.

**Lab Question 1:**
* (a) How did the Poincaré plot *change* when you used more points?
* (b) What does this tell you about the importance of choosing a small `dt` (high `Pts per Cycle`) when simulating a chaotic system?

---

### Experiment 1: Order (Periodic Motion)

1.  Set the sliders to "tame" values:
    * **Drive (F_D): 0.5**
    * **Pts per Cycle: 100** (a good default)
2.  Click **"Run Simulation"**.

**What to Observe:**
* **Time Series Plot:** A simple, smooth, repeating wave.
* **FFT Plot:** A single, sharp "spike" at the drive frequency, $f_D$.
* **Phase Space Plot:** A simple, closed loop (a "limit cycle").
* **Poincaré Plot:** A **single dot**.

**Lab Question 2:**
* (a) Why does a **single dot** in the Poincaré plot prove the motion is periodic?
* (b) How does this single dot relate to the simple loop in the Phase Space plot?

---

### Experiment 2: The "Route to Chaos" (Period Doubling)

This is the most important part.
1.  Increase the drive strength slightly:
    * **Drive (F_D): 1.07**
2.  Click **"Run Simulation"**.

**What to Observe:**
* **Time Series Plot:** A repeating pattern, but it now takes *two* drive cycles to repeat (a "big swing, little swing").
* **FFT Plot:** You'll see the original $f_D$ peak, plus a new, large peak exactly at **$f_D/2$** (the sub-harmonic).
* **Poincaré Plot:** You will now see **TWO dots**.

3.  Now, turn the knob a little more:
    * **Drive (F_D): 1.082**
4.  Click **"Run Simulation"**.

**What to Observe:**
* **Poincaré Plot:** You should now see **FOUR dots**.
* **FFT Plot:** You'll see new peaks appear at **$f_D/4$** and **$3f_D/4$**.

**Lab Question 3:**
* (a) How do the "two dots" (Poincaré) and the new "$f_D / 2$" peak (FFT) both tell you that the period has **doubled**?
* (b) What did you see in the plots at $F_D=1.082$ that proves the period doubled *again*?

---

### Experiment 3: The Strange Attractor (Full Chaos)

Let's push the system over the edge.
1.  Set the drive strength to the chaotic value:
    * **Drive (F_D): 1.2**
2.  Click **"Run Simulation"**.

**What to Observe:**
* **Time Series Plot:** A wild, erratic, non-repeating "scribble."
* **FFT Plot:** The sharp peaks are gone, replaced by a "grassy," noisy **broadband spectrum**.
* **Phase Space Plot:** The lines scribble to fill a complex area.
* **Poincaré Plot:** A complex, fractal-like pattern of dots. This is the **Strange Attractor**.

**Lab Question 4:**
* Explain how the "cloud of points" (Poincaré plot) and the "broadband noise" (FFT plot) are two different "fingerprints" telling you the system is chaotic.

---

### Experiment 4: The Butterfly Effect

This experiment shows why chaos is unpredictable.
1.  Run a simulation with these exact values:
    * **Drive (F_D): 1.2**
    * **Start θ (deg): 0.0**
2.  Look closely at the **Time Series Plot**. Remember its pattern.
3.  Now, make one *tiny* change:
    * **Start θ (deg): 0.1** (a 0.1-degree difference!)
4.  Run the simulation again.

**What to Observe:**
* At first, the **Time Series Plot** will look identical to the previous run.
* But after 10-15 time units, you'll see the two patterns **completely diverge**.

**Lab Question 5 (The Big One):**
* (a) Do the **Poincaré plots** for the $\theta=0.0$ and $\theta=0.1$ runs look different from each other?
* (b) Based on your answers, can we predict the pendulum's *exact* position $\theta$ a long time in the future? Can we predict the *overall pattern of states* (the "shape" of its motion) it will visit? Explain this apparent contradiction.

### Experiment 5: The Speed Test!

We've just built a custom solver from scratch. Was it worth the effort?

In computational physics, speed is critical. A simulation that takes 10 seconds is fun; one that takes 10 minutes is not. We're going to use a standard profiling tool, `%timeit`, to benchmark our solver against the "black box" `solve_ivp` from SciPy.

**The Test:**
The cell below will:
1.  Define a function `run_scipy` that runs the full simulation using `solve_ivp`.
2.  Define a function `run_numba_rk4` that runs the *exact same* simulation using our new `@jit` compiled `rk4_step` solver.
3.  Use `%timeit` to run both functions multiple times and get a reliable average speed for each.

**Your Task:**
1.  Run the cell below.
2.  Wait for the `%timeit` results to print. The first run (for Numba) will include the one-time "compilation" cost, but the average time will be very fast.
3.  Look at the final times (e.g., "1.2 s per loop" vs "15 ms per loop") and answer the final lab question.

In [ ]:
# We need to re-import time for this cell
import time
# --- ADD THIS HELPER FUNCTION ---
# This is the fast, JIT-compiled solver loop that run_numba_rk4 calls.

@jit(nopython=True)
def solve_with_rk4(y0_np, q, F_D, w_D, T_D, dt,
                       n_points, start_index, points_per_cycle):
    """
    Runs the full simulation with our fast RK4 solver.
    """
    # Create empty arrays to store the results
    t_vals = np.zeros(n_points)
    theta_vals = np.zeros(n_points)
    omega_vals = np.zeros(n_points)

    # Poincaré points (max of N_cycles points)
    n_poincare_max = n_points // points_per_cycle + 1
    poincare_theta = np.zeros(n_poincare_max)
    poincare_omega = np.zeros(n_poincare_max)

    # Set initial state
    y = y0_np
    t = 0.0
    poincare_count = 0

    # The main solver loop
    for i in range(n_points):
        # Store the current state
        t_vals[i] = t
        theta_vals[i] = y[0]
        omega_vals[i] = y[1]

        # Check if this is a Poincaré point (strobe flash)
        if i >= start_index and i % points_per_cycle == 0:
            if poincare_count < len(poincare_theta): # Avoid array overflow
                poincare_theta[poincare_count] = y[0]
                poincare_omega[poincare_count] = y[1]
                poincare_count += 1

        # Calculate the next step
        y = rk4_step(t, y, dt, q, F_D, w_D)
        t += dt

    # Return only the points we actually saved
    return (t_vals, theta_vals, omega_vals,
            poincare_theta[:poincare_count], poincare_omega[:poincare_count])
# --- Version 1: SciPy's solve_ivp ---
# We have to wrap it in a function for timeit to test
def run_scipy():
    # Standard setup for 500 cycles
    q, F_D, w_D = 1.2, 0.667, 0.5
    y0 = [np.deg2rad(0.0), 0.0]
    T_D = 2.0 * np.pi / w_D
    t_max = 500 * T_D
    t_span = [0, t_max]

    # We must use t_eval to force solve_ivp to do the same amount of work
    points_per_cycle = 100
    n_points = 500 * points_per_cycle + 1
    t_eval = np.linspace(0, t_max, n_points)

    sol = solve_ivp(
        derivs, # This is the Numba-jitted derivs, but solve_ivp can't use it fast
        t_span,
        y0,
        args=(q, F_D, w_D),
        t_eval=t_eval,
        method='RK45'
    )
    return sol.y # Return the result so the work isn't optimized away

# --- Version 2: Our Numba+RK4 solver ---
# We re-use the fast functions you already wrote in Cell 2
# @jit(nopython=True) def derivs(...)
# @jit(nopython=True) def rk4_step(...)

def run_numba_rk4():
    # Standard setup for 500 cycles
    q, F_D, w_D = 1.2, 0.667, 0.5
    y0_np = np.array([np.deg2rad(0.0), 0.0])
    T_D = 2.0 * np.pi / w_D

    points_per_cycle = 100
    dt = T_D / points_per_cycle
    n_points = 500 * points_per_cycle + 1

    # We must wrap the call to the *inner* numba loop
    # that we defined in Cell 4
    t_v, th_v, o_v, p_th, p_om = solve_with_rk4(
        y0_np, q, F_D, w_D, T_D, dt,
        n_points, 0, points_per_cycle
    )
    return th_v # Return the result

# --- Run the timing experiment ---
# NOTE: We run each function *once* first to "warm up" numba
# (i.e., let it finish compiling)
print("Warming up Numba JIT compiler...")
_ = run_numba_rk4()
print("...Warmup complete. Starting benchmark.\n")

print("Timing SciPy's solve_ivp:")
# %timeit -n 1 -r 3 means "run it 1 time, repeat this 3 times"
# This is slow, so we don't do many loops
%timeit -n 1 -r 3 run_scipy()

print("\nTiming our Numba + RK4 solver:")
# This is fast, so we can do more loops for a better average
%timeit -n 5 -r 7 run_numba_rk4()

Warming up Numba JIT compiler...
...Warmup complete. Starting benchmark.

Timing SciPy's solve_ivp:
721 ms ± 5.67 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)

Timing our Numba + RK4 solver:
31.9 ms ± 1.31 ms per loop (mean ± std. dev. of 7 runs, 5 loops each)
